In [1]:
import warnings
warnings.filterwarnings("ignore")

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local-cluster[3,2,2048]")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(sc.getConf().get("spark.driver.memory", "NOT SET"))

2g


In [2]:
from pyspark.sql.functions import *
spark.sparkContext.setJobDescription('read data (csv)')
df_csv = spark.read.format('csv').option('inferSchema', 'true').option('header', 'true').load('yellow_tripdata_combined.csv')

group_csv = df_csv.groupBy('passenger_count').agg(sum('fare_amount'))

spark.sparkContext.setJobDescription('groupby ---> write data (csv)')
group_csv.write.format('noop').mode('overwrite').save()

In [3]:
spark.sparkContext.setJobDescription('read data (parquet)')
df_par = spark.read.format('parquet').load('yellow_tripdata_combined.parquet')

group_par = df_par.groupBy('passenger_count').sum('fare_amount')

spark.sparkContext.setJobDescription('groupby ---> write data (parquet)')
group_par.write.format('noop').mode('overwrite').save()